# 🧠 NeuroSynk - Pipeline Oficial de Entrenamiento en TensorFlow con GPU
### Clasificación Continua de Estados Cognitivos (6,979 Muestras Unificadas)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josedanielbal09-hue/Neurosynk/blob/main/NeuroSynk_Entrenamiento_TensorFlow_Oficial.ipynb)

Este cuaderno oficial entrena la **Red Neuronal Profunda de NeuroSynk** en TensorFlow / Keras y la exporta en formato **TensorFlow.js (`model.json` y `weights.bin`)** para ejecución 100% privada en el navegador.

In [ ]:
# 1. Instalación Limpia (Google Colab ya tiene TensorFlow y GPU preinstalados)
!pip install -q --no-deps tensorflowjs
print("✅ Entorno verificado y listo sin conflictos de paquetes.")

In [ ]:
# 2. Verificación de TensorFlow y GPU
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
import glob

print(f"🟢 TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"⚡ GPU Activa en Google: {gpus if gpus else 'CPU Activa'}")

In [ ]:
# 3. Carga Automática del Dataset (Detecta cualquier CSV subido o pide seleccionarlo)
df = None

# A. Buscar si ya existe algún archivo CSV en el entorno
candidate_files = glob.glob('*.csv') + glob.glob('/content/*.csv')
for f in candidate_files:
    try:
        temp = pd.read_csv(f)
        if 'ear_mean' in temp.columns or 'label' in temp.columns:
            df = temp
            print(f"✅ Dataset detectado y cargado automáticamente desde: {f}")
            break
    except Exception:
        continue

# B. Si no se encuentra, abrir selector interactivo de carga
if df is None:
    from google.colab import files
    print("📂 Por favor, sube tu archivo CSV (dataset_unificado_total_neurosynk.csv o dataset_ventanas_aumentado.csv):")
    uploaded = files.upload()
    for fn in uploaded.keys():
        df = pd.read_csv(fn)
        print(f"✅ Archivo subido cargado con éxito: {fn}")
        break

print(f"\n📊 Total de Muestras Cargadas: {len(df)}")
if 'label_name' in df.columns:
    print("\nDistribución por Clase:")
    print(df['label_name'].value_counts())
df.head()

In [ ]:
# 4. Normalización Z-Score de las Características Biométricas
all_possible_features = [
    'ear_mean', 'ear_min', 'yaw_mean', 'yaw_std', 'pitch_mean', 'pitch_std',
    'frown_mean', 'nose_delta_sum', 'gaze_variance_mean', 'shoulder_angle_mean',
    'mar_mean', 'roll_angle_mean'
]

# Asegurar que todas las 12 características existan en el dataframe
for col in all_possible_features:
    if col not in df.columns:
        df[col] = 0.0

feature_cols = all_possible_features
X = df[feature_cols].values.astype(np.float32)
y = df['label'].values.astype(np.int32)

means = np.mean(X, axis=0)
stds = np.std(X, axis=0)
stds[stds == 0] = 1.0

X_norm = (X - means) / stds
num_classes = 6
y_one_hot = tf.keras.utils.to_categorical(y, num_classes=num_classes)

print(f"Tensor de Entrada X: {X_norm.shape} | Tensor de Salida y: {y_one_hot.shape}")

In [ ]:
# 5. Arquitectura Neuronal Profunda y Entrenamiento en GPU (80 Épocas)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(feature_cols),), name='biometric_input'),
    
    # Capa Densa 1
    tf.keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.15),
    
    # Capa Densa 2
    tf.keras.layers.Dense(32, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.10),
    
    # Capa Densa 3
    tf.keras.layers.Dense(16, activation='relu', kernel_initializer='he_normal'),
    
    # Capa de Salida Softmax
    tf.keras.layers.Dense(num_classes, activation='softmax', name='cognitive_state')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

print("\n🚀 Entrenando Red Neuronal en GPU...")
history = model.fit(
    X_norm, y_one_hot,
    epochs=80,
    batch_size=16,
    validation_split=0.15,
    shuffle=True
)

# Graficar Curvas de Aprendizaje Oficiales
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento', color='#10b981', lw=2)
plt.plot(history.history['val_accuracy'], label='Validación', color='#38bdf8', lw=2)
plt.title('Precisión (Accuracy) de la Red Neuronal')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()
plt.grid(alpha=0.2)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento', color='#f59e0b', lw=2)
plt.plot(history.history['val_loss'], label='Validación', color='#ef4444', lw=2)
plt.title('Pérdida (Cross-Entropy Loss)')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# 6. Exportación en Formato TensorFlow.js Web y Descarga Automática
import tensorflowjs as tfjs
from google.colab import files

out_dir = 'neurosynk_model_tfjs'
os.makedirs(out_dir, exist_ok=True)
tfjs.converters.save_keras_model(model, out_dir)

class_names = [
    'ESTUDIO NORMAL / NEUTRO',
    'ENFOQUE PROFUNDO (FLOW)',
    'DISTRACCIÓN',
    'FATIGA',
    'SOBREESTIMULACIÓN',
    'AGOBIO POSTURAL'
]

metadata = {
    'featureMeans': means.tolist(),
    'featureStds': stds.tolist(),
    'classNames': class_names,
    'accuracy': f"{history.history['val_accuracy'][-1]*100:.1f}%",
    'loss': f"{history.history['val_loss'][-1]:.4f}",
    'epochs': 80,
    'samplesCount': len(X),
    'featureNames': feature_cols
}

with open(f'{out_dir}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("\n🎉 ¡MODELO EXPORTADO CON ÉXITO PARA LA WEB!")
!zip -r neurosynk_model_tfjs.zip neurosynk_model_tfjs
print("Descargando neurosynk_model_tfjs.zip...")
files.download('neurosynk_model_tfjs.zip')